In [1]:
from Run_Designs import design_experiments, design_initial_experiments
from methods.DataPrep import dataprep
import numpy as np
import pandas as pd

Design Variable

1. C - List of number of categroies corresponding to each categorical variable
2. Nc - Number of categorical variables
3. Nx - Continuous variables
4. h - categorical
5. x - continuous
6. z - [h, x]
7. nDim - Nc + Nx
8. bounds - Lower and Upper bounds of both Catergorical variables and continuous variables
9. Var_unsc - Variables that shouldn't be scaled in the model

Sample Collection Details

10. n_iter - Number of iterations to run the algorithm
11. initN - Number of intial data points
12. batch_size (b) - Number of experiments to be generated in each iteration

Optimization Problem details
13. approach_type (type of design space)
    - Single system/task learning - CoCa (Continuous Categorical) vs Co (Continuous)
    - Multi system/task learning - CoCa_ICM, CoCa_LCM, CoCa_LEK, CoCa_LEK_precompute, 
    Co_ICM, Co_LCM, Co_LEK, Co_LEK_precompute
    
14. prob_type - 'UnConstrained' vs 'Constrained' (type of optimization problem)
15. method - 'thompson sampling' vs 'constant_liar' (type of sampling)
16. Meas_Noise - Measurement Noise in the target 
17. trade_off : Explorarion exploitation balance

Multitask/TL details

17. Noutput : Number of different systems in the data. Needed for multi-task or transfer learning application (System ID is always in the last column of the dataset)
18. if approach type for multi-task is ICM or LCM --> W_rank is a required parameter for the model. if approach type is LEK based then N_latent is required. Default W_rank = 1. Default N_latent = 2
19.



In [2]:
Stock_solid = pd.read_excel('CarbonSourceInfo.xlsx', 'Stocks_solid')
Stock_liquid = pd.read_excel('CarbonSourceInfo.xlsx', 'Stocks_liquid')

Carbon_Names = Stock_solid['Carbon Source'].values.tolist()
Carbon_Names.append(Stock_liquid['Carbon Source'][1])
Carbon_Names.append(Stock_liquid['Carbon Source'][2])

Carbon_Ub = [50]*17 #[g/L]
Carbon_Ub.append(10) # Glycerol [mL/L]
Carbon_Ub.append(50) #Ethanol [mL/L]

In [3]:
Mol_List = ['KPhaffii_HSA'] #, 'KPhaffii_RBDJ', 'Kphaffii_Trastuzumab'
Data_Molecules = {}
for m in Mol_List:
    Data_Molecules[m] = pd.read_excel("KPhaffii_Experiments.xlsx", m).iloc[:,1:]
    for i in range(Data_Molecules[m].shape[0]):
        Data_Molecules[m].iloc[i,0] = Carbon_Names.index(Data_Molecules[m].iloc[i,0])
        Data_Molecules[m].iloc[i,1] = Data_Molecules[m].iloc[i,1]/Carbon_Ub[Data_Molecules[m].iloc[i,0]]
        Data_Molecules[m].iloc[i,2:4] = Data_Molecules[m].iloc[i,2:4]/10
        
data_modified_np = {}
result_modified_np = {}
cnt_m = 0
for m in Mol_List:
    data_modified = []
    result_modified = []
    data_init = Data_Molecules[m].iloc[:,0:4].values.astype('float64')
    result_init = Data_Molecules[m]['SP [mg/L/OD600]'].values.reshape(-1,1).astype('float64')
    for nc in range(data_init.shape[0]):
#         if data_init[nc, 1] == 0:
#             temp = np.concatenate((np.arange(0, 19).reshape(-1, 1), np.zeros((19, 1)),
#                                    np.multiply(data_init[nc, 2:], np.ones((19, 2)))), axis=1)
#             temp_res = result_init[nc] * np.ones((19, 1))
#             data_modified + temp.tolist()
#             result_modified + temp_res.tolist()
#         else:
        data_modified.append(data_init[nc, :])
        result_modified.append(result_init[nc])
    
    temp1 = np.array(data_modified)
    data_modified_np[m] = np.concatenate((temp1, cnt_m*np.ones((temp1.shape[0], 1))), axis =1)
    result_modified_np[m] = np.array(result_modified).reshape(-1,1)
    
    cnt_m = cnt_m+1

In [4]:
data = data_modified_np['KPhaffii_HSA']
result = result_modified_np['KPhaffii_HSA']
Nround = 4
path = '../Exp/RBDJ/'


for r in range(Nround):
    data_modified = []
    result_modified = []
    dat_read = pd.read_csv("./Round"+str(r)+'/Reconstructed_Round'+str(r)+"_CoCaLEK.csv").iloc[:,1:].values #Reconstructed_
    result_new = pd.read_csv(path+"Round" +str(r)+"_CoCaLEK/Round" + str(r) +"_Result_Summary_final.csv")['Specific Productivity'].values.reshape(-1,1)

    for nc in range(dat_read.shape[0]):
#         if dat_read[nc, 1] == 0:
#             temp = np.concatenate((np.arange(0, 19).reshape(-1, 1), np.zeros((19, 1)),
#                                    np.multiply(dat_read[nc, 2:], np.ones((19, 2)))), axis=1)
#             temp_res = result_new[nc] * np.ones((19, 1))
#             data_modified = data_modified + temp.tolist()
#             result_modified = result_modified + temp_res.tolist()
#         else:
        data_modified.append(dat_read[nc, :])
        result_modified.append(result_new[nc])
    
    temp1 = np.array(data_modified)
    data_new = np.concatenate((temp1, 1*np.ones((temp1.shape[0], 1))), axis =1)
    result_new_np = np.array(result_modified).reshape(-1,1)
    
    
    data = np.concatenate((data, data_new), axis = 0)
    result = np.concatenate((result, result_new_np), axis = 0)

In [5]:
import pickle
fnl = "./Round3/3_ModelBackground_CoCaLEK.pkl" #_updated
with open(fnl, "rb") as fp:
    ModelBackground = pickle.load(fp)
    
key = 'CoCa_LEK'
data_param_CoCaLEK = ModelBackground['data_param']
data_mod = dataprep(data, data_param_CoCaLEK, key)

In [6]:
method = 'constant_liar'
batch_size = 11
output_file_name = './Round4/Round4_CoCaLEK'
background_model_parameter = './Round3/3_ModelBackground_CoCaLEK.pkl' #_updated
new_model_parameter = './Round4/4_ModelBackground_CoCaLEK.pkl'

z_next, gp_actual = design_experiments(data_mod, result,batch_size, method,background_model_parameter, 
                            new_model_parameter , output_file_name)

[[ 7]
 [ 0]
 [ 0]
 [15]
 [ 0]
 [ 8]
 [ 0]
 [ 2]
 [ 9]
 [ 0]
 [ 0]]
[array([163.41923679,   1.        ,   1.50688251,   1.        ,
         1.        ,   1.        ,   1.        ,   1.92561288,
         2.21401689,   2.28878389,   1.        ,   1.        ,
         1.        ,   1.        ,   1.        ,   1.81688151,
         1.        ,   1.        ,   1.        ])]
I am here
  mul.                                           |              value  |  constraints  |  priors
  sum.MixtureViaSumAndProduct.mix                |  0.616467707783104  |      +ve      |        
  sum.MixtureViaSumAndProduct.Mat52.lengthscale  |               (3,)  |      +ve      |        
  sum.white.variance                             |                0.1  |   +ve fixed   |        
  latentVariable.Wembedding                      |             (2, 2)  |               |        
0
(5000, 4)
1
[8.         0.7462982  0.08200989 0.11279196 0.         1.        ]
1
(5000, 4)
1
[1.60000000e+01 1.58755243e-01 5.94025

In [ ]:
Carbon_Names[12]

In [ ]:
# import pickle
# fnl = "./Round0/0_ModelBackground_CoCaLEK.pkl"
# with open(fnl, "rb") as fp:
#     ModelBackground = pickle.load(fp)

In [ ]:
# ModelBackground['Categorical_dist_param']['ht_batch_list'] = dat_read[:,0:1].tolist()

In [ ]:
# import pickle
# with open("./Round0/0_ModelBackground_CoCaLEK_updated.pkl", 'wb') as output:
#     # Pickle dictionary using protocol 0.
#     pickle.dump(ModelBackground, output)

In [ ]:
embed = ModelBackground['Categorical_dist_param']['Wembed_updated']

In [ ]:
from matplotlib import pyplot as plt
plt.scatter(embed[:,0], embed[:,1])

In [ ]:
np.atleast_2d(ModelBackground['Categorical_dist_param']['ht_batch_list']).shape

In [ ]:
ModelBackground['Categorical_dist_param'].keys()

In [ ]:
ModelBackground['Categorical_dist_param']['Wc_list']